# 04 — Transcriptómica espacial

**Taller de célula única CIAD**

Adaptado de la viñeta *Analysis of spatial datasets* de Seurat:
<https://satijalab.org/seurat/articles/spatial_vignette>

Todo hasta ahora perdía una cosa: **dónde estaba la célula**. El tejido se
disociaba en una suspensión, y la posición se perdía antes de empezar a secuenciar.

La transcriptómica espacial la conserva. Mides la expresión *y* sabes de qué parte del
tejido vino cada medición.

**Los datos.** Un portaobjetos Visium de 10x de un cerebro de ratón, corte sagital, mitad
anterior. El tejido se coloca sobre un portaobjetos impreso con ~5,000 spots en una
cuadrícula regular, cada uno de 55 µm de ancho. Cada spot captura el RNA que tiene encima, etiquetado con un
barcode que identifica su posición. Primero se toma una fotografía del tejido teñido,
para poder poner la expresión de regreso sobre la anatomía.

**Un límite importante.** Un spot de 55 µm es más grande que una célula. Cada spot
contiene más o menos de 1 a 10 células, y lo que mides es su mezcla. Así que un "spot" no
es una célula, y esto no son datos de célula única. Son datos bulk con una posición,
medidos sobre un área muy pequeña. Todo lo demás se sigue de eso.

**Qué hacemos**

1. cargar los datos, y ver cómo se ve una imagen dentro de un objeto Seurat
2. control de calidad, que aquí se comporta distinto
3. normalizar con `SCTransform`
4. graficar genes conocidos sobre la anatomía
5. hacer clustering **sin ninguna información espacial**, y después graficar los clusters sobre el
   tejido y ver qué pasó

Unos 30 minutos.

## Preparación

Este notebook usa su propio archivo de preparación. Agrega `glmGamPoi`, que hace
que `SCTransform` sea mucho más rápido — los otros notebooks no usan `SCTransform`, así que
no lo instalan.

In [ ]:
# This cell installs the packages we need. It is a shortcut for the workshop,
# so that nobody spends the class waiting for an install.
source("https://raw.githubusercontent.com/MartinLoza/CIAD_workshop_sc/main/setup/setup_spatial.R")

In [ ]:
# Change R language to English, in case your computer uses another language.
Sys.setenv(LANGUAGE = "en")

# This is the normal way to load a package in R. You will write lines like
# these at the top of every script you make.
library(Seurat)
library(ggplot2)
library(dplyr)
library(patchwork)

# Size of every figure in this notebook, in inches. Change these two numbers
# if a plot looks too small or too large.
options(repr.plot.width = 10, repr.plot.height = 7)

## 1. Cargar un conjunto de datos Visium

Dos archivos, y son de tipo distinto:

- **`filtered_feature_bc_matrix.h5`** — los counts. Spots x genes, en HDF5, que
  es la razón por la que nuestra preparación instala `hdf5r`.
- **`spatial.tar.gz`** — la fotografía del tejido, más una tabla con las
  coordenadas en píxeles de cada spot en esa fotografía.

`Load10X_Spatial()` espera que estén acomodados en un solo directorio, con los archivos de imagen
en una subcarpeta `spatial/` — el acomodo que produce Space Ranger. Nosotros lo recreamos.

In [ ]:
base <- "https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Mouse_Brain_Sagittal_Anterior"
h5   <- "V1_Mouse_Brain_Sagittal_Anterior_filtered_feature_bc_matrix.h5"

dir.create("visium", showWarnings = FALSE)

In [ ]:
download.file(file.path(base, h5), file.path("visium", h5), quiet = TRUE)
download.file(file.path(base, "V1_Mouse_Brain_Sagittal_Anterior_spatial.tar.gz"),
              "spatial.tar.gz", quiet = TRUE)

untar("spatial.tar.gz", exdir = "visium")

list.files("visium", recursive = TRUE)

Mira lo que salió del tarball:

- `tissue_lowres_image.png` — la fotografía, a menor resolución
- `tissue_positions_list.csv` — una fila por spot: su barcode, si está sobre
  tejido, y sus coordenadas de fila/columna y de píxeles
- `scalefactors_json.json` — cómo convertir entre los píxeles de resolución completa y la
  imagen de menor resolución

Esa es toda la parte espacial. El resto es una matriz de counts como cualquier otra.

In [ ]:
brain <- Load10X_Spatial(data.dir = "visium", filename = h5)

brain

### Qué es distinto en este objeto

Compara ese resumen con el del notebook 01. Cambiaron dos cosas.

El assay se llama **`Spatial`**, no `RNA` — es una convención de nombres, nada más profundo.

Y el objeto ahora carga una **imagen**. Eso es nuevo: `Images()` las enlista, y
las coordenadas de cada spot vienen con ella.

In [ ]:
cat("assay      :", DefaultAssay(brain), "\n")
cat("spots      :", ncol(brain), "\n")
cat("genes      :", nrow(brain), "\n")
cat("images     :", Images(brain), "\n\n")

coords <- GetTissueCoordinates(brain)
cat("coordinates, first 5 spots:\n")
head(coords, 5)

Dos números por spot, y son posiciones en píxeles sobre la fotografía. Esa es
toda la diferencia entre esto y los datos del notebook 01.

Los nombres de los spots son barcodes, exactamente como antes.

In [ ]:
head(colnames(brain), 3)

head(brain@meta.data, 3)

## 2. Control de calidad, y por qué aquí es distinto

`nCount_Spatial` es el total de counts por spot. En el notebook 02 una gota con pocos counts era
una célula muriendo o una gota vacía, y la quitábamos.

Aquí, pocos counts pueden significar simplemente **menos células debajo de ese spot**. Una región de
tejido poco denso da pocos counts porque así es la anatomía, no
porque algo haya salido mal.

Así que mira los counts *en el espacio* antes de decidir nada.

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 6)

p1 <- VlnPlot(brain, features = "nCount_Spatial", pt.size = 0.1) + NoLegend()
p2 <- SpatialFeaturePlot(brain, features = "nCount_Spatial") +
  theme(legend.position = "right")

p1 + p2

El panel derecho es todo el argumento. Los counts varían de forma suave a lo largo del
corte y el patrón sigue la anatomía — las estructuras densas dan counts altos,
los ventrículos y los tractos de fibras dan counts bajos.

Filtra eso con un umbral global y borras anatomía, no datos malos.

### ✏️ Ejercicio 1

¿En qué parte del corte están los counts más bajos, y eso es un problema técnico o
biológico?

No hay que programar nada. Mira el panel derecho y di qué harías.

*Tu respuesta:*

## 3. Normalización

La varianza en estos datos depende de cuántas células había debajo de cada spot, lo cual
varía de forma sistemática a lo largo del tejido. La log-normalización simple asume que esa
variación es técnica y la quita — aquí, parte de ella es anatomía.

`SCTransform` modela los counts de cada gen con una binomial negativa regularizada y
normaliza sin esa suposición. Es la recomendación de la viñeta para
datos espaciales, y la razón por la que este notebook instala `glmGamPoi`.

Un minuto, más o menos.

In [ ]:
brain <- SCTransform(brain, assay = "Spatial", verbose = FALSE)

Assays(brain)

Un assay nuevo, `SCT`, ahora el que está por defecto. `Spatial` sigue teniendo los counts crudos.

## 4. Genes sobre la anatomía

Esta es la parte que hace útiles a los datos espaciales.

`SpatialFeaturePlot()` pone la expresión de regreso sobre la fotografía. Dos genes con
expresión muy localizada y bien conocida en el cerebro de ratón:

- **`Hpca`** — hipocalcina, prácticamente restringida al hipocampo
- **`Ttr`** — transtiretina, producida por el plexo coroideo, una estructura pequeña en los
  ventrículos

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 6)

SpatialFeaturePlot(brain, features = c("Hpca", "Ttr"))

Sin clustering, sin anotación, sin referencia. Dos genes graficados sobre una fotografía,
y el hipocampo y el plexo coroideo aparecen solos.

### Hacer legibles las gráficas

Dos argumentos son los que más importan. `pt.size.factor` escala los spots — más grande
llena el tejido, más chico deja ver la imagen de abajo. `alpha` hace transparentes los spots
con poca expresión para que el patrón resalte.

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 6)

p1 <- SpatialFeaturePlot(brain, features = "Ttr", pt.size.factor = 1) +
  ggtitle("small spots")
p2 <- SpatialFeaturePlot(brain, features = "Ttr", alpha = c(0.1, 1)) +
  ggtitle("faded where low")

p1 + p2

### ✏️ Ejercicio 2

Grafica otros dos genes con expresión regional fuerte en el cerebro de ratón.
Sugerencias: `Mbp` (mielina, es decir materia blanca), `Calb1` (calbindina), `Pcp4`
(proteína 4 de células de Purkinje), `Nrgn` (corteza).

Llena el espacio en blanco:

In [ ]:
# SpatialFeaturePlot(brain, features = c(______, ______))

## 5. Clustering

Esta es la parte importante de este notebook.

Ahora corremos **exactamente el pipeline del notebook 02** — PCA, vecinos, clusters,
UMAP. Nada de eso sabe nada de la posición. Las coordenadas espaciales no se le pasan
a ninguna de estas funciones. Para el clustering, estas son 2,700
muestras sin relación entre sí.

In [ ]:
brain <- RunPCA(brain, assay = "SCT", verbose = FALSE)
brain <- FindNeighbors(brain, dims = 1:30, verbose = FALSE)
brain <- FindClusters(brain, resolution = 0.8, verbose = FALSE)
brain <- RunUMAP(brain, dims = 1:30, verbose = FALSE)

table(Idents(brain))

Ahora grafica esos clusters dos veces: en el UMAP, y sobre el tejido.

In [ ]:
options(repr.plot.width = 16, repr.plot.height = 7)

p1 <- DimPlot(brain, reduction = "umap", label = TRUE) + NoLegend() +
  ggtitle("UMAP — no spatial information used")
p2 <- SpatialDimPlot(brain, label = TRUE, label.size = 3) +
  ggtitle("the same clusters, on the tissue")

p1 + p2

**Ese es el resultado de este notebook.**

Los clusters se calcularon solo a partir de la expresión. Nunca se le dio la posición al
algoritmo. Aun así, al graficarlos de regreso sobre el corte reconstruyen la anatomía del cerebro —
las capas corticales como bandas apiladas, el hipocampo como una curva distinta, los tractos de materia
blanca separados de la materia gris.

Esto dice algo fuerte sobre los datos. Las células de una misma región del tejido son lo bastante
parecidas en su expresión como para que el clustering encuentre la región sin que le digan que
existe.

También es una revisión de tu análisis. Si los clusters aparecieran como puntitos
dispersos por todo el corte, algo estaría mal — el clustering o el
tejido.

### Un cluster a la vez

Los colores encimados son difíciles de leer. `SpatialDimPlot()` puede resaltar
clusters individuales en lugar de todos.

In [ ]:
options(repr.plot.width = 16, repr.plot.height = 8)

SpatialDimPlot(
  brain,
  cells.highlight = CellsByIdentities(brain, idents = c(1, 2, 3, 5)),
  facet.highlight = TRUE,
  ncol = 4
)

### ✏️ Ejercicio 3

Escoge un cluster que forme una región compacta y encuentra qué lo marca.

Llena los espacios en blanco:

In [ ]:
# my_cluster <- ______
#
# markers <- FindMarkers(brain, ident.1 = my_cluster, only.pos = TRUE, verbose = FALSE)
# head(markers, 5)
#
# SpatialFeaturePlot(brain, features = rownames(markers)[1])

## 6. Marcadores de una región espacial

El mismo `FindMarkers()` del notebook 02. Lo que cambió es que el resultado se puede
graficar sobre la anatomía, así que un marcador se puede revisar a ojo.

In [ ]:
cluster_markers <- FindMarkers(brain, ident.1 = 1, only.pos = TRUE, verbose = FALSE)

head(cluster_markers, 5)

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 6)

SpatialFeaturePlot(brain, features = rownames(cluster_markers)[1:2])

## Qué hicimos, y qué dejamos fuera

De un portaobjetos a la anatomía etiquetada:

- cargamos counts más una imagen en un solo objeto
- vimos que el control de calidad tiene que tomar en cuenta la estructura del tejido, no solo la falla técnica
- normalizamos con `SCTransform`
- graficamos genes sobre la fotografía y reconocimos estructuras a partir de dos genes
- hicimos clustering sin información espacial, y aun así recuperamos la anatomía

**Dejado fuera a propósito**, todo está en la viñeta si lo quieres:

- **Varios cortes.** Los cortes se integran de forma muy parecida a los batches del notebook
  03 — la maquinaria que ya tienes.
- **Transferencia de etiquetas desde datos de célula única.** Como un spot son varias células, un
  siguiente paso natural es estimar qué tipos celulares componen cada spot, usando una
  referencia de scRNA-seq anotada y `FindTransferAnchors()`. Necesita un conjunto de datos de
  referencia grande y bastante tiempo.
- **Genes con variación espacial.** `FindSpatiallyVariableFeatures()` encuentra genes con
  patrón espacial sin usar clusters, con la I de Moran. La idea es buena, pero en la práctica es
  lenta.
- **Plataformas basadas en imagen** — Xenium, MERFISH, CosMx — que sí son
  de célula única y vienen con las fronteras de las células. Se cubren en una
  [viñeta aparte](https://satijalab.org/seurat/articles/seurat5_spatial_vignette_2).

El punto principal: los datos espaciales se analizan con las herramientas que ya aprendiste. Lo nuevo es que cada resultado se puede poner de regreso sobre el tejido y
revisarlo contra la anatomía — que es una forma de validación mucho más fuerte que un UMAP.

---

### Respuestas

<details>
<summary>Clic para desplegar</summary>

**Ejercicio 1**

Los counts más bajos están sobre los tractos de fibras y los ventrículos. Eso es biológico: debajo de esos
spots hay menos células, y células con menos RNA. Quitar los spots con pocos counts
borraría esas estructuras del corte.

Para datos espaciales, el control de calidad apunta mejor a la falla *técnica* — spots fuera del
tejido, o un borde dañado del corte — que encuentras mirando la
imagen, no poniendo un umbral a los counts. `Load10X_Spatial()` ya quitó
por nosotros los spots que están fuera del tejido.

**Ejercicio 2**

```r
SpatialFeaturePlot(brain, features = c("Mbp", "Nrgn"))
```

`Mbp` marca la materia blanca y `Nrgn` la corteza, así que los dos son casi
complementarios.

**Ejercicio 3**

```r
my_cluster <- 3
markers <- FindMarkers(brain, ident.1 = my_cluster, only.pos = TRUE, verbose = FALSE)
head(markers, 5)
SpatialFeaturePlot(brain, features = rownames(markers)[1])
```

Si el patrón de expresión del mejor marcador coincide con la posición del cluster en el
corte, el cluster es una región real. Si no coincide, desconfía.

</details>